In [ ]:
# ==========================================
# FEATURE ENGINEERING
# ==========================================

import pandas as pd
import numpy as np

# -----------------------------------
# Load Datasets
# -----------------------------------

print("Loading datasets...")

fraud = pd.read_csv("../data/raw/Fraud_Data.csv")

country = pd.read_csv(
    "../data/raw/IpAddress_to_Country.csv"
)

print("Fraud Shape:", fraud.shape)
print("Country Shape:", country.shape)

# -----------------------------------
# Remove Duplicates
# -----------------------------------

fraud = fraud.drop_duplicates()

print("Duplicates removed")

# -----------------------------------
# Datetime Conversion
# -----------------------------------

fraud["signup_time"] = pd.to_datetime(
    fraud["signup_time"]
)

fraud["purchase_time"] = pd.to_datetime(
    fraud["purchase_time"]
)

print("Datetime conversion completed")

# ===================================
# FEATURE ENGINEERING
# ===================================

# -----------------------------------
# Time Since Signup
# -----------------------------------

fraud["time_since_signup_hours"] = (
    fraud["purchase_time"]
    - fraud["signup_time"]
).dt.total_seconds() / 3600

print("time_since_signup_hours completed")

# -----------------------------------
# Hour of Day
# -----------------------------------

fraud["hour_of_day"] = (
    fraud["purchase_time"]
    .dt.hour
)

print("hour_of_day completed")

# -----------------------------------
# Day of Week
# -----------------------------------

fraud["day_of_week"] = (
    fraud["purchase_time"]
    .dt.dayofweek
)

print("day_of_week completed")

# ===================================
# OPTIONAL TRANSACTION FEATURES
# ===================================

# Faster than groupby-transform

print("Creating transaction count features...")

user_counts = fraud["user_id"].value_counts()

fraud["user_transaction_count"] = (
    fraud["user_id"].map(user_counts)
)

device_counts = fraud["device_id"].value_counts()

fraud["device_transaction_count"] = (
    fraud["device_id"].map(device_counts)
)

print("Transaction count features completed")

# ===================================
# IP ADDRESS PROCESSING
# ===================================

print("Processing IP addresses...")

fraud["ip_address"] = (
    fraud["ip_address"]
    .astype(float)
    .astype(np.int64)
)

country["lower_bound_ip_address"] = (
    country["lower_bound_ip_address"]
    .astype(np.int64)
)

country["upper_bound_ip_address"] = (
    country["upper_bound_ip_address"]
    .astype(np.int64)
)

print("IP conversion completed")

# ===================================
# GEOLOCATION MERGE
# ===================================

print("Sorting datasets...")

fraud = fraud.sort_values(
    "ip_address"
).reset_index(drop=True)

country = country.sort_values(
    "lower_bound_ip_address"
).reset_index(drop=True)

print("Starting country merge...")

merged = pd.merge_asof(
    fraud,
    country,
    left_on="ip_address",
    right_on="lower_bound_ip_address",
    direction="backward"
)

print("Filtering valid ranges...")

merged = merged[
    merged["ip_address"]
    <= merged["upper_bound_ip_address"]
]

print("Country merge completed")

# ===================================
# COUNTRY FRAUD ANALYSIS
# ===================================

country_fraud = (
    merged.groupby("country")["class"]
    .mean()
    .sort_values(ascending=False)
)

print("\nTop Fraud Countries")
print(country_fraud.head(10))

# ===================================
# SAVE PROCESSED DATA
# ===================================

merged.to_csv(
    "../data/processed/fraud_processed.csv",
    index=False
)

print("\nFeature Engineering Completed Successfully")
print("Final Shape:", merged.shape)

# ===================================
# FEATURE CHECK
# ===================================

print("\nGenerated Features")

generated_features = [
    "time_since_signup_hours",
    "hour_of_day",
    "day_of_week",
    "user_transaction_count",
    "device_transaction_count"
]
for feature in generated_features:
    print(feature)


Loading datasets...
Fraud Shape: (151112, 11)
Country Shape: (138846, 3)
Duplicates removed
Datetime conversion completed
time_since_signup_hours completed
hour_of_day completed
day_of_week completed
Creating transaction count features...
Transaction count features completed
Processing IP addresses...
IP conversion completed
Sorting datasets...
Starting country merge...
Filtering valid ranges...
Country merge completed

Top Fraud Countries
country
Turkmenistan             1.000000
Namibia                  0.434783
Sri Lanka                0.419355
Luxembourg               0.388889
Virgin Islands (U.S.)    0.333333
Ecuador                  0.264151
Tunisia                  0.262712
Peru                     0.260504
Bolivia                  0.245283
Kuwait                   0.233333
Name: class, dtype: float64

Feature Engineering Completed Successfully
Final Shape: (129146, 19)

Generated Features
time_since_signup_hours
hour_of_day
day_of_week
user_transaction_count
device_transaction_